In [ ]:
### 교차 검증과 그리드 서치

# 판다스 라이브러리로 외부 데이터(와인) 호출
import pandas as pd
wine = pd.read_csv('https://bit.ly/wine_csv_data')
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']




# 훈련-타깃 데이터 세트 분리
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size = 0.2, random_state = 42)

#훈련 데이터를 재차 훈련 세트(sub)와 검증 세트(val)로 분리
sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target, test_size = 0.2, random_state = 42)
print("훈련 세트와 검증 세트 크기:", sub_input.shape, val_input.shape) # 훈련 세트와 검증 세트를 약 4:1 비율로 추출.



# 결정 트리 알고리즘 - 훈련 세트와 검증 세트 정확도 비교
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state = 42)
dt.fit(sub_input, sub_target)
print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target)) # 훈련 정확도 >>> 검증 정확도; 과대적합



# 사이킷런 함수를 활용한 교차 검증

from sklearn.model_selection import cross_validate # k-fold cross validation 함수
scores = cross_validate(dt, train_input, train_target) # 매개변수: '모델 객체', '훈련 세트 입력', '훈련 세트 타깃'
print(scores) # 딕셔너리 반환; fit_time: 훈련 시간 / score_time: 검증 시간 / test_score: "검증 폴드" 점수

import numpy as np
from sklearn.model_selection import cross_val_score # k-fold 교차 검증 결과만을 출력하는 메서드
print("cross_val_score를 통한 5-폴드 교차 검증 결과:", np.round(cross_val_score(dt, train_input, train_target), decimals = 4))
print("5-fold 교차 검증 결과 평균: ", np.mean(scores['test_score'])) # Dictionary 문법 - 특정 key에 해당하는 Value만 반환 가능



# 폴드 교차 검증 이전 데이터 셔플을 위한 분할기 설정 - 분류 모델의 경우
from sklearn.model_selection import StratifiedKFold
# 분류 모델의 경우 => cv = StratifiedKFold() 매개변수 지정을 통해 훈련 세트 셔플
scores = cross_validate(dt, train_input, train_target, cv = StratifiedKFold())
print(np.mean(scores['test_score']))



# k 횟수를 늘린 10-폴드 교차 검증 수행 결과
splitter = StratifiedKFold(n_splits = 10, shuffle = True, random_state = 42) # n_splits는 분할기가 사용하는 속성.
# 검증 횟수와 훈련 세트 셔플 여부를 포함한 분할기 => cross_validate 함수의 파라미터로 사용됨.
scores = cross_validate(dt, train_input, train_target, cv = splitter)
print("10-fold 교차 검증 결과 평균: ", np.mean(scores['test_score']))



# 하이퍼파라미터 튜닝을 위한 그리드 서치 객체 생성
from sklearn.model_selection import GridSearchCV
params = {'min_impurity_decrease' : [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]} # min_impurity_decrease => 결정 트리 알고리즘의 매개변수!!!
gs = GridSearchCV(DecisionTreeClassifier(random_state = 42), params, n_jobs = -1)
# GridSerachCV('탐색 대상 모델, 모델 매개변수, 사용할 CPU 코어 수) 지정
gs.fit(train_input, train_target)

print("그리드 서치 기반 하이퍼파라미터 튜닝 후 정확도:", dt.score(train_input, train_target))
print("하이퍼파라미터 최적값: ", gs.best_params_) # .best_params => 하이퍼파라미터 최적값을 담은 딕셔너리
print("각 하이퍼파라미터 값에 따른 훈련 정확도: ", gs.cv_results_['mean_test_score']) # .cv_result_ => 그리드 서치 결과를 담은 딕셔너리?
print(".cv_results_에는 무엇이 담겨있는가? => ", gs.cv_results_)
print("cv_results에서 추출한 하이퍼파라미터 최적값: ", gs.cv_results_['params'][gs.best_index_])




# 두 개 이상의 하이퍼파라미터 튜닝 - 결정 트리 알고리즘의 max_depth & min_impurity_decrease & min_samples_split

params = {'min_impurity_decrease' : np.arange(0.0001, 0.001, 0.0001), 'max_depth' : np.arange(5, 20, 1), 'min_samples_split': np.arange(2, 100, 10)}
# np.arange(a, b, c) => a에서 출발하여 b에 도달할 때까지 c를 더한 값을 담은 배열 형성

gs = GridSearchCV(DecisionTreeClassifier(random_state = 42), params, n_jobs = -1)
gs.fit(train_input, train_target)
print(gs.best_params_) # 최적의 깊이 제한 & 불순도 감소 최소치 & 최소 샘플 수 조합 도출
print(np.max(gs.cv_results_['mean_test_score'])) # 세 하이퍼파라미터 조합이 최적일 때 정확도



### 랜덤 서치(그리드 서치 외 튜닝 기법) - 하이퍼파라미터 4개 튜닝

from scipy.stats import uniform, randint
# scipy를 통해 임의(랜덤)값을 input으로 하이퍼파라미터 튜닝
params = {'min_impurity_decrease' : uniform(0.0001, 0.001), 'max_depth' : randint(20, 50), 'min_samples_split': randint(2, 25), 'min_samples_leaf': randint(1, 25)}

from sklearn.model_selection import RandomizedSearchCV
rs = RandomizedSearchCV(DecisionTreeClassifier(random_state = 42), params, n_iter = 100, n_jobs = -1, random_state = 42)
rs.fit(train_input, train_target)

print(rs.best_params_)
print(np.max(rs.cv_results_['mean_test_score']))

dt = rs.best_estimator_
print(dt.score(test_input, test_target))

In [ ]:
### 그리드 서치와 랜덤 서치의 작동 시간 비교 - time 라이브러리 사용

# 판다스 라이브러리로 외부 데이터(와인) 호출
import pandas as pd
wine = pd.read_csv('https://bit.ly/wine_csv_data')
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']


# 훈련-타깃 데이터 세트 분리
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size = 0.2, random_state = 42)


from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state = 42)
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from scipy.stats import uniform, randint
import time

params = {'min_impurity_decrease' : np.arange(0.0001, 0.001, 0.0001), 'max_depth' : np.arange(5, 20, 1), 'min_samples_split': np.arange(2, 100, 10)}

gs = GridSearchCV(DecisionTreeClassifier(random_state = 42), params, n_jobs = -1)
gs_bf = time.time()
gs.fit(train_input, train_target)
gs_aft = time.time()
print("그리드 서치 훈련 소요 시간: ", gs_aft - gs_bf)

params = {'min_impurity_decrease' : uniform(0.0001, 0.001), 'max_depth' : randint(20, 50), 'min_samples_split': randint(2, 25)}

rs = RandomizedSearchCV(DecisionTreeClassifier(random_state = 42), params, n_iter = 100, n_jobs = -1, random_state = 42)
rs.bf = time.time()
rs.fit(train_input, train_target)
rs.aft = time.time()
print("랜덤 서치 훈련 소요 시간: ", rs.aft - rs.bf)

In [ ]:
### 랜덤 서치 알고리즘을 활용한 경사하강법 하이퍼파라미터(학습률, 반복 횟수) 튜닝

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from scipy.stats import uniform, randint
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline # Pipeline import 추가

# 1. 외부 데이터 호출 및 훈련-테스트 세트 추출
fish = pd.read_csv('https://bit.ly/fish_csv_data')
fish_input = fish[['Weight', 'Length', 'Diagonal', 'Height', 'Width']]
fish_target = fish['Species']

train_input, test_input, train_target, test_target = train_test_split(fish_input, fish_target, random_state=42)

# sgd_before를 위한 스케일러 적용 (train_scaled, test_scaled 정의)
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_input)
test_scaled = scaler.transform(test_input)

# 2. 튜닝 전 모델 성능 확인
sgd_before = SGDClassifier(loss='log_loss', max_iter=10, random_state=42, tol=None)
sgd_before.fit(train_scaled, train_target)
print(f"튜닝 전 테스트 정확도: {sgd_before.score(test_scaled, test_target)}")



# 3. 파이프라인 구성(전처리 과정에서 발생 가능한 데이터 누설 방지)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('sgd', SGDClassifier(
        loss='log_loss',
        learning_rate='constant',
        random_state=42,
        tol=None
    ))
])



# 4. 랜덤 서치 하이퍼파라미터 튜닝
# learning_rate='constant' 설정 => 학습률 매개변수: eta0

# 파이프라인 내의 SGDClassifier의 파라미터는 'sgd__파라미터명' 형식으로 지정
params = {'sgd__eta0': uniform(0.0001, 0.1), 'sgd__max_iter': randint(100, 10000)}

rs = RandomizedSearchCV(
    pipe, # 모델 객체로 pipe 전달
    params,
    n_iter=100,
    n_jobs=-1,
    random_state=42
)
rs.fit(train_input, train_target)




# 5. 최적의 모델 확인 및 평가
print("최적의 하이퍼파라미터:", rs.best_params_)
best_model = rs.best_estimator_
print(f"튜닝 후 테스트 정확도: {best_model.score(test_input, test_target)}")